# Preprocess v_1: làm sạch điểm GPS và chia dev/holdout

- **Tham số:** [`config.yaml`](config.yaml). **Code của từng bước:** `src/geolife/preprocess.py`. Lý do chọn từng ngưỡng: [`docs/preprocess_v1.md`](../../docs/preprocess_v1.md).
- Mỗi bước in số điểm / file bị ảnh hưởng và vài dòng mẫu để kiểm tra.
- Preprocess **không** chứa giả định của bài toán home/office (khung đêm, ngày làm việc...). Các giả định đó thuộc về model.

| Bước | Việc làm |
| --- | --- |
| 1 | Đọc raw |
| 2 | Bỏ toạ độ / năm không hợp lệ, bỏ trùng timestamp |
| 3 | Tìm file trùng giữa các user (chỉ đánh dấu) |
| 4 | Loại file chuyến bay |
| 5 | Loại file lỗi nặng |
| 6 | Xoá spike A → B → A |
| 7 | Xoá "đảo": đoạn ngắn bị kẹp bởi bước nhảy > 400 km/h |
| 8 | Giờ local theo timezone thật của toạ độ |
| 9 | User gần trùng (cùng chỗ, cùng lúc) → nhóm user |
| 10 | Chia dev / holdout theo nhóm, kiểm tra rò rỉ |
| 11 | Ghi output |

In [1]:
import json, time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from geolife import io, preprocess as pp
from geolife.geo import haversine
from geolife.paths import RAW, preprocess_dir
from geolife.timezone import lookup_tz, to_local, tz_country

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "axes.titleweight": "bold"})

cfg = yaml.safe_load(open("config.yaml"))
OUT = preprocess_dir(cfg["version"])
OUT.mkdir(parents=True, exist_ok=True)
rep, t0 = {"version": cfg["version"], "steps": {}}, time.time()


def log(step, **kw):
    # Ghi số liệu của một bước vào report.json và in ra
    rep["steps"][step] = kw
    print(step, kw, "| còn", f"{len(pts):,}", "điểm")

## 1. Đọc raw

Mỗi file `.plt` có 6 dòng header, rồi các cột `lat, lon, 0, altitude, days, date, time`. Chỉ giữ `lat, lon, date + time`.
Altitude bị bỏ vì bài toán không cần và cột này không đáng tin (EDA mục 3). Nhãn phương tiện (`labels.txt`, 69 user) được đọc riêng,
**không gắn vào điểm** trong output: model không được nhìn thấy nhãn, nhãn chỉ dùng để kiểm tra chéo ở đây và để đánh giá.

In [2]:
pts, labels, files = io.read_raw(RAW)
files["traj_key"] = files.user.astype(str) + "_" + files.traj_id
rep["raw"] = {"points": len(pts), "trajs": len(files), "users": int(files.user.nunique()), "label_rows": len(labels)}
print(rep["raw"])
display(pts.sample(4, random_state=0)); display(labels.head(3))

{'points': 24876978, 'trajs': 18670, 'users': 182, 'label_rows': 14718}


,user,traj_id,ts_utc,lat,lon
6263122,25,20090311021122,2009-03-11 02:33:17,39.976012,116.310625
10314132,41,20090712044248,2009-07-12 06:57:44,39.865115,118.533055
22110569,153,20111015094803,2011-10-15 10:08:05,40.046160,116.399788
16645900,124,20080930203247,2008-10-01 01:19:39,34.099994,92.355716


,user,start_utc,end_utc,mode
0,10,2007-06-26 11:32:29,2007-06-26 11:40:29,bus
1,10,2008-03-28 14:52:54,2008-03-28 15:59:59,train
2,10,2008-03-28 16:00:00,2008-03-28 22:02:00,train


## 2. Giá trị không hợp lệ và trùng timestamp

- Toạ độ ngoài miền xác định, hoặc năm ngoài 2007–2012 (thời gian thu thập của GeoLife): xoá.
- Trùng `(user, ts_utc)`: giữ điểm đầu. 99,5% điểm trùng giây cách điểm đầu < 10 m (EDA mục 3), nên giữ điểm nào cũng như nhau.
  Phải bỏ trước khi tính tốc độ, vì dt = 0 làm tốc độ thành vô hạn.

In [3]:
v = cfg["validity"]
bad = pp.invalid_mask(pts, v["lat_range"], v["lon_range"], v["years"])
print("Điểm không hợp lệ:"); display(pts[bad])
pts = pts[~bad].sort_values(["user", "ts_utc"], kind="stable").reset_index(drop=True)
dup = pp.duplicate_timestamp_mask(pts)
print("Ví dụ trùng timestamp (dòng thứ 2 bị xoá):")
i = np.flatnonzero(dup)[0]; display(pts.iloc[i - 1:i + 1])
pts = pts[~dup].reset_index(drop=True)
pts["traj_key"] = pts.user.astype(str) + "_" + pts.traj_id
pts = pp.add_kinematics(pts)
pts["mode"] = pp.label_points(pts, labels)   # chỉ để kiểm tra chéo trong notebook, không ghi ra output
log("2_validity", invalid=int(bad.sum()), duplicate_timestamp=int(dup.sum()), dt_le_0_left=int((pts.dt_s <= 0).sum()))

Điểm không hợp lệ:


,user,traj_id,ts_utc,lat,lon
5067886,20,20110911000506,2011-09-11 03:23:54,400.166667,116.215390
22578366,163,20000101231219,2000-01-01 23:12:19,39.988992,116.327023
22578367,163,20000101231219,2000-01-01 23:13:21,39.990964,116.327041
22578368,163,20000101231219,2000-01-01 23:15:23,39.993207,116.326827


Ví dụ trùng timestamp (dòng thứ 2 bị xoá):


,user,traj_id,ts_utc,lat,lon
311761,2,20081101010123,2008-11-01 01:09:31,39.895867,116.272799
311762,2,20081101010123,2008-11-01 01:09:31,39.895870,116.272962


2_validity {'invalid': 4, 'duplicate_timestamp': 698900, 'dt_le_0_left': 0} | còn 24,178,074 điểm


## 3. File trùng giữa các user

Một file được coi là trùng khi cùng tên, cùng số điểm, cùng điểm đầu và cùng thời điểm cuối (EDA mục 4).
**Không xoá** (không biết ai là chủ thật), chỉ ghi lại để bước 9–10 xếp các user này vào cùng một nhóm.

In [4]:
files, dup_pairs = pp.duplicate_file_pairs(files)
print(f"{files.is_cross_dup.sum():,} file trùng, {files.n_raw[files.is_cross_dup].sum():,} điểm, {len(dup_pairs)} cặp user")
print("Cặp user có nhiều file chung nhất:"); display(dup_pairs.nlargest(5, "shared_files"))

1,679 file trùng, 2,966,117 điểm, 44 cặp user
Cặp user có nhiều file chung nhất:


,user_a,user_b,shared_files
43,153,163,225
39,126,167,145
0,0,3,129
2,11,88,59
40,128,153,55


## 4. File chuyến bay

**Quy tắc:** loại cả file khi median tốc độ của file > 300 km/h. Điểm đang bay không thể là một lần dừng.

**Bằng chứng:** phân bố median tốc độ theo file có một khoảng trống: không file nào nằm trong 250–450 km/h, nên mọi ngưỡng trong
khoảng này cho cùng một kết quả. Kiểm tra chéo bằng nhãn `airplane` / `train`.

In [5]:
g = pts.groupby("traj_key", sort=False)
ts = pd.DataFrame({"n": g.size(), "median_speed_kmh": g.speed_kmh.median(),
                   "air": g["mode"].agg(lambda s: (s == "airplane").mean()),
                   "train": g["mode"].agg(lambda s: (s == "train").mean())})
air = ts.air >= 0.3
print(pd.cut(ts.median_speed_kmh, [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 700, 900, 2000]).value_counts().sort_index().to_string())
print("Median của file có nhãn airplane:", sorted(ts.median_speed_kmh[air].round().astype(int)))
print("Median lớn nhất của file có nhãn train:", round(ts.median_speed_kmh[ts.train >= 0.5].max(), 1))
for thr in [250, 300, 450, 800]:
    sel = ts.median_speed_kmh > thr
    print(f"ngưỡng {thr}: {sel.sum()} file, bắt {(sel & air).sum()}/{air.sum()} file airplane")
flight = ts.index[ts.median_speed_kmh > cfg["flight"]["traj_median_speed_kmh"]]
display(ts.loc[flight].sort_values("median_speed_kmh").head(5).round(2))
pts = pp.drop_trajs(pts, flight)
log("4_flight", trajs=len(flight), points=int(ts.n[flight].sum()), airplane_labeled=int(air[flight].sum()))

median_speed_kmh
(0, 50]        17379
(50, 100]       1137
(100, 150]       102
(150, 200]        13
(200, 250]         6
(250, 300]         0
(300, 350]         0
(350, 400]         0
(400, 450]         0
(450, 500]         1
(500, 700]         6
(700, 900]        12
(900, 2000]        0
Median của file có nhãn airplane: [6, 663, 686, 787, 815, 852, 868]
Median lớn nhất của file có nhãn train: 206.0
ngưỡng 250: 19 file, bắt 6/7 file airplane
ngưỡng 300: 19 file, bắt 6/7 file airplane
ngưỡng 450: 19 file, bắt 6/7 file airplane
ngưỡng 800: 8 file, bắt 3/7 file airplane


,n,median_speed_kmh,air,train
traj_key,,,,
144_20090321083221,9078,474.18,0.0,0.0
128_20100927102732,1014,525.81,0.0,0.0
128_20100523030239,465,554.63,0.0,0.0
144_20081109052637,682,637.17,0.0,0.0
128_20090222040348,447,662.89,1.0,0.0


4_flight {'trajs': 19, 'points': 72863, 'airplane_labeled': 6} | còn 24,105,211 điểm


## 5. File lỗi nặng

**Spike A → B → C:** tốc độ A→B và B→C đều > 150 km/h, và C quay về gần A: d(A, C) < 0,3 · d(A, B).
Bảng độ nhạy dưới đây chọn 150 / 0,3: lõi khoảng 600 spike không đổi khi tăng ngưỡng tới 300 km/h; hạ xuống 100 km/h thì số file
bị ảnh hưởng tăng vọt, tức là bắt đầu chạm vào chuyển động thật.

**File lỗi nặng:** spike ≥ 1% **và** ≥ 10 spike. Chỉ dùng tỉ lệ thì phạt file ngắn (1 spike / 5 điểm = 20%); chỉ dùng số đếm thì
phạt file dài. Phân bố số spike theo file cho thấy file lỗi tách hẳn khỏi phần còn lại.

In [6]:
rows = []
for vv in [100, 150, 200, 300]:
    for rr in [0.1, 0.3, 0.5]:
        s = pp.find_spikes(pts, vv, rr)
        rows.append({"v_kmh": vv, "ratio": rr, "spikes": s.sum(), "files": pts.traj_key[s].nunique(),
                     "labeled_train_airplane": pts["mode"][s].isin(["train", "airplane"]).sum()})
display(pd.DataFrame(rows).pivot(index="v_kmh", columns="ratio", values=["spikes", "files", "labeled_train_airplane"]))

sp, cc = cfg["spike"], cfg["corrupted_traj"]
s = pp.find_spikes(pts, sp["speed_kmh"], sp["return_ratio"])
spk = pd.DataFrame({"spikes": pd.Series(pts.traj_key[s]).value_counts()})
spk["n"] = ts.n.reindex(spk.index); spk["ratio"] = spk.spikes / spk.n
print(pd.cut(spk.spikes, [0, 1, 2, 5, 10, 100, 1e6]).value_counts().sort_index().to_string())
r_or = (spk.ratio >= cc["min_spike_ratio"]) | (spk.spikes >= cc["min_spikes"])
r_and = (spk.ratio >= cc["min_spike_ratio"]) & (spk.spikes >= cc["min_spikes"])
print("OR :", spk[r_or].round(4).to_dict("index"))
print("AND:", spk[r_and].round(4).to_dict("index"))
corrupted = spk.index[r_and]
pts = pp.drop_trajs(pts, corrupted)
log("5_corrupted", trajs=list(corrupted), points=int(spk.n[r_and].sum()))

spikes            files           labeled_train_airplane        
ratio    0.1  0.3   0.5   0.1  0.3  0.5                    0.1 0.3 0.5
v_kmh                                                                 
100      677  951  1196    83  293  467                      2   3   3
150      624  710   767    32  109  154                      2   2   2
200      604  620   630    13   28   38                      0   0   0
300      594  595   595     3    4    4                      0   0   0

spikes
(0.0, 1.0]            99
(1.0, 2.0]             6
(2.0, 5.0]             2
(5.0, 10.0]            0
(10.0, 100.0]          1
(100.0, 1000000.0]     1
OR : {'62_20080926000623': {'spikes': 577, 'n': 8091, 'ratio': 0.0713}, '62_20081003000004': {'spikes': 16, 'n': 4271, 'ratio': 0.0037}, '153_20091129234104': {'spikes': 3, 'n': 252, 'ratio': 0.0119}, '128_20070620130412': {'spikes': 1, 'n': 66, 'ratio': 0.0152}, '153_20080614021333': {'spikes': 1, 'n': 69, 'ratio': 0.0145}, '167_20080512103623': {'spikes': 1, 'n': 5, 'ratio': 0.2}}
AND: {'62_20080926000623': {'spikes': 577, 'n': 8091, 'ratio': 0.0713}}


5_corrupted {'trajs': ['62_20080926000623'], 'points': 8091} | còn 24,097,120 điểm


## 6. Xoá spike

Áp dụng 150 km/h / 0,3, lặp tối đa 3 lần (xoá một spike có thể làm lộ spike kế bên).

In [7]:
s = pp.find_spikes(pts, sp["speed_kmh"], sp["return_ratio"])
i = np.flatnonzero(s)[0]
print("Ví dụ (dòng giữa là B, bị xoá):")
display(pts.iloc[i - 1:i + 2][["traj_key", "ts_utc", "lat", "lon", "dt_s", "dist_m", "speed_kmh"]].round({"lat": 5, "lon": 5, "dist_m": 1, "speed_kmh": 1}))
print(f"d(A, C) = {haversine(pts.lat.iat[i-1], pts.lon.iat[i-1], pts.lat.iat[i+1], pts.lon.iat[i+1]):.0f} m")
pts, removed = pp.remove_spikes(pts, sp["speed_kmh"], sp["return_ratio"], sp["max_iter"])
log("6_spike", removed_per_iter=removed)

Ví dụ (dòng giữa là B, bị xoá):


,traj_key,ts_utc,lat,lon,dt_s,dist_m,speed_kmh
9653,0_20081115010133,2008-11-15 04:00:55,40.00974,116.32226,5.0,12.5,9.0
9654,0_20081115010133,2008-11-15 04:01:05,40.01378,116.32034,10.0,477.6,171.9
9655,0_20081115010133,2008-11-15 04:01:10,40.01018,116.32173,5.0,417.8,300.8


d(A, C) = 66 m


6_spike {'removed_per_iter': [133, 0]} | còn 24,096,987 điểm


## 7. Điểm còn tốc độ > 400 km/h: xoá điểm không có liên kết hợp lý, giữ bước nhảy giữa hai đoạn dài

**Ngưỡng 400 km/h:** phương tiện mặt đất có nhãn gần như không vượt (bảng đầu); ngưỡng thấp hơn bắt đầu chạm điểm tàu / xe thật.

**Vấn đề của cách xoá theo điểm:** xoá mọi điểm có tốc độ vào > 400 rồi lặp lại gây xoá dây chuyền: điểm kế tiếp lại thành "quá tốc độ".
Tốc độ là thuộc tính của **liên kết giữa hai điểm**, không phải của một điểm. Câu hỏi đúng là: ở hai phía của bước nhảy, phía nào sai?

**Quy tắc:** cắt mỗi file thành các **đoạn** tại bước nhảy > 400 km/h và tại khoảng không ghi > 5 phút. Bên trong một đoạn, các điểm
nối nhau bằng liên kết hợp lý, tức là tự nhất quán.
- Đoạn chỉ có **1 điểm** mà có ít nhất một đầu là bước nhảy là **đảo**: điểm đó không có liên kết hợp lý nào với hàng xóm. Xoá.
  Gồm: điểm lạc đơn lẻ (vào nhanh, ra nhanh), đoạn bay trong file hỗn hợp (chuỗi điểm lạc liên tiếp), fix sai ngay khi bật máy
  hoặc ngay sau khi mất tín hiệu.
- Bước nhảy giữa hai đoạn **có từ 2 điểm**: cả hai phía đều tự nhất quán nên **giữ cả hai** (thường là mất tín hiệu rồi bắt lại,
  ví dụ vào hầm hoặc trong nhà). Vị trí không sai; chỉ tốc độ của đúng một liên kết là vô nghĩa, và output không chứa cột tốc độ.

Bảng 2: số đoạn bị bước nhảy chặn ít nhất một đầu, theo số điểm. Bảng 3: thời lượng của các đoạn đó, để thấy vì sao chỉ xoá đoạn 1 điểm.
Bảng 4: độ nhạy theo ngưỡng khoảng không ghi.

In [8]:
r5 = {"lat": 5, "lon": 5, "dt_s": 0, "dist_m": 1, "speed_kmh": 1}
mode_s = pts["mode"].astype(str).to_numpy(); vi = pts.speed_kmh.to_numpy()
display(pd.DataFrame([{"threshold": x, "links": (vi > x).sum(), "labeled_train": ((vi > x) & (mode_s == "train")).sum(),
                       "labeled_car_bus_taxi": ((vi > x) & np.isin(mode_s, ["car", "bus", "taxi"])).sum()}
                      for x in [250, 300, 400, 500]]))
isl = cfg["island"]
seg_tab, seg = pp.segments(pts, isl["speed_kmh"], isl["gap_s"])
jumped = seg_tab[seg_tab.fast_before | seg_tab.fast_after].copy()
jumped["ends"] = np.where(jumped.fast_before & jumped.fast_after, "bước nhảy ở 2 đầu", "bước nhảy ở 1 đầu")
jumped["size"] = pd.cut(jumped.n_points, [0, 1, 2, 3, 5, 10, 50, 1e9], labels=["1", "2", "3", "4-5", "6-10", "11-50", ">50"])
display(jumped.pivot_table(index="size", columns="ends", values="n_points", aggfunc="size", observed=False, fill_value=0))
display(jumped.groupby("size", observed=True).dur_s.describe(percentiles=[.5, .9])[["count", "50%", "90%", "max"]])
display(pd.DataFrame([{"gap_s": g, "islands": int(pp.island_mask(pts, isl["speed_kmh"], g, isl["max_points"])[0].sum())}
                      for g in [60, 300, 600, 1800, 1e12]]))

,threshold,links,labeled_train,labeled_car_bus_taxi
0,250,9277,80,160
1,300,8024,51,120
2,400,6697,17,73
3,500,5850,14,51


ends,bước nhảy ở 1 đầu,bước nhảy ở 2 đầu
size,,
1,305,5252
2,65,13
3,29,7
4-5,49,9
6-10,84,19
11-50,393,58
>50,1599,77


,count,50%,90%,max
size,,,,
1,5557.0,0.0,0.0,0.0
2,78.0,5.0,70.3,245.0
3,36.0,24.5,218.5,290.0
4-5,58.0,8.0,157.5,596.0
6-10,103.0,60.0,337.2,660.0
11-50,451.0,172.0,908.0,2396.0
>50,1676.0,1125.0,3592.5,33681.0


,gap_s,islands
0,6.000000e+01,5921
1,3.000000e+02,5557
2,6.000000e+02,5445
3,1.800000e+03,5346
4,1.000000e+12,5260


In [9]:
mask, seg_tab = pp.island_mask(pts, isl["speed_kmh"], isl["gap_s"], isl["max_points"])
cols = ["traj_key", "ts_utc", "lat", "lon", "dt_s", "dist_m", "speed_kmh", "mode"]
fast = pts.speed_kmh.to_numpy() > isl["speed_kmh"]
prev_gap = np.r_[pts.dt_s.to_numpy()[1:] > isl["gap_s"], False]   # điểm i đứng ngay sau khoảng không ghi
i = np.flatnonzero(mask & fast)[0]
print("Ví dụ điểm lạc (bị xoá):"); display(pts.iloc[i - 1:i + 3][cols].round(r5))
i = np.flatnonzero(mask & ~fast & np.r_[fast[1:], False] & (pts.dt_s.to_numpy() > isl["gap_s"]))[0]
print("Ví dụ fix sai ngay sau khoảng không ghi (dòng thứ 2, bị xoá):"); display(pts.iloc[i - 1:i + 3][cols].round(r5))
n_mode = pts["mode"][mask].value_counts().to_dict()
pts = pp.drop_points(pts, mask)
left = np.flatnonzero(pts.speed_kmh.to_numpy() > isl["speed_kmh"])
print("Ví dụ bước nhảy giữa hai đoạn dài (giữ):"); display(pts.iloc[left[0] - 2:left[0] + 3][cols].round(r5))
log("7_island", removed=int(mask.sum()), removed_labeled=n_mode, jumps_kept=len(left),
    jumps_kept_users=int(pts.user.iloc[left].nunique()))

Ví dụ điểm lạc (bị xoá):


,traj_key,ts_utc,lat,lon,dt_s,dist_m,speed_kmh,mode
1984820,10_20071228075610,2007-12-28 09:06:52,40.12872,116.59083,2.0,217.4,391.2,NaN
1984821,10_20071228075610,2007-12-28 09:06:55,40.13153,116.59239,3.0,339.1,406.9,NaN
1984822,10_20071228075610,2007-12-28 09:06:57,40.13337,116.59357,2.0,228.0,410.4,NaN
1984823,10_20071228075610,2007-12-28 09:06:58,40.13428,116.59422,1.0,115.0,413.8,NaN


Ví dụ fix sai ngay sau khoảng không ghi (dòng thứ 2, bị xoá):


,traj_key,ts_utc,lat,lon,dt_s,dist_m,speed_kmh,mode
9642,0_20081115010133,2008-11-15 03:36:37,40.00984,116.32113,5.0,8.8,6.3,NaN
9643,0_20081115010133,2008-11-15 04:00:05,40.00246,116.31132,1408.0,1171.3,3.0,NaN
9644,0_20081115010133,2008-11-15 04:00:13,40.00994,116.32273,8.0,1279.5,575.8,NaN
9645,0_20081115010133,2008-11-15 04:00:15,40.00995,116.32273,2.0,0.6,1.1,NaN


Ví dụ bước nhảy giữa hai đoạn dài (giữ):


,traj_key,ts_utc,lat,lon,dt_s,dist_m,speed_kmh,mode
13932,0_20090330005208,2009-03-30 10:10:20,40.01435,116.32389,5.0,27.2,19.6,NaN
13933,0_20090330005208,2009-03-30 10:10:25,40.01480,116.32416,5.0,55.8,40.2,NaN
13934,0_20090330005208,2009-03-30 10:10:30,40.00908,116.32246,5.0,653.1,470.2,NaN
13935,0_20090330005208,2009-03-30 10:10:35,40.00964,116.32248,5.0,62.5,45.0,NaN
13936,0_20090330005208,2009-03-30 10:10:40,40.00960,116.32252,5.0,5.7,4.1,NaN


7_island {'removed': 5557, 'removed_labeled': {'walk': 17, 'bus': 7, 'bike': 4, 'train': 2, 'car': 2, 'airplane': 1, 'taxi': 1}, 'jumps_kept': 1148, 'jumps_kept_users': 116} | còn 24,091,430 điểm


## 8. Giờ local theo timezone thật của từng toạ độ

Timestamp gốc là UTC. Timezone được tra bằng `timezonefinder` (ranh giới timezone IANA) cho **từng toạ độ duy nhất**
(làm tròn 4 chữ số, ~11 m), rồi đổi giờ theo đúng timezone đó tại đúng thời điểm, nên có DST (ví dụ `America/Los_Angeles` có cả −7 và −8).
Không có giả định "mọi điểm là giờ Trung Quốc".

Output giữ `tz`. Quốc gia (để tra lịch nghỉ lễ của đúng nước đó) được suy ra từ `tz` ở bước model, không gắn ở đây.

In [10]:
t1 = time.time()
pts["tz"] = lookup_tz(pts.lat.to_numpy(), pts.lon.to_numpy(), cfg["timezone"]["coord_round"])
pts["ts_local"] = to_local(pts.ts_utc, pts.tz)
tzc = pts.tz.value_counts()
tab = pd.DataFrame({"points": tzc, "share_%": (tzc / len(pts) * 100).round(3), "country": tz_country(tzc.index)})
display(tab[tab.points > 0])
off = (pts.ts_local - pts.ts_utc).dt.total_seconds() / 3600
print("Offset ở America/Los_Angeles:", off[pts.tz == "America/Los_Angeles"].value_counts().to_dict())
display(pts[pts.tz != "Asia/Shanghai"].sample(4, random_state=1)[["user", "lat", "lon", "tz", "ts_utc", "ts_local"]])
log("8_timezone", n_tz=int((tzc > 0).sum()), unknown=int((pts.tz == "Unknown").sum()),
    no_country_points=int(tab.points[tab.country == ""].sum()), seconds=round(time.time() - t1, 1))

,points,share_%,country
tz,,,
Asia/Shanghai,23366770,96.992,CN
America/Los_Angeles,222926,0.925,US
Asia/Taipei,71796,0.298,TW
Europe/Rome,60948,0.253,IT
Asia/Urumqi,54546,0.226,CN
Europe/Paris,51853,0.215,FR
Europe/Madrid,36772,0.153,ES
Asia/Hong_Kong,30347,0.126,HK
Asia/Phnom_Penh,28238,0.117,KH


Offset ở America/Los_Angeles: {-7.0: 149094, -8.0: 73832}


,user,lat,lon,tz,ts_utc,ts_local
20039724,153,43.699627,7.268245,Europe/Paris,2009-04-29 17:09:37,2009-04-29 19:09:37
19207685,144,8.190029,98.296358,Asia/Bangkok,2009-03-22 07:55:18,2009-03-22 14:55:18
22729796,163,42.357936,-71.091826,America/New_York,2012-04-11 21:23:45,2012-04-11 17:23:45
20090818,153,41.876728,12.518696,Europe/Rome,2009-05-07 05:58:42,2009-05-07 07:58:42


8_timezone {'n_tz': 26, 'unknown': 0, 'no_country_points': 12, 'seconds': 5.0} | còn 24,091,430 điểm


## 9. User gần trùng: cùng chỗ, cùng lúc

Khoá file ở bước 3 chỉ bắt file giống hệt. Hai user vẫn có thể chia sẻ dữ liệu theo cách khác (file bị cắt khác, hai thiết bị đi cùng nhau).
Nếu một user ở `dev` và user kia ở `holdout`, tham số chỉnh trên `dev` sẽ "nhìn thấy trước" holdout: **rò rỉ dữ liệu**.

**Cách đo:** chia thời gian thành phút và không gian thành ô ~100 m. `share` của một cặp = số (phút, ô) chung / số (phút, ô) của user
ít dữ liệu hơn. Phân bố `share` dưới đây dùng để chọn ngưỡng gộp nhóm.

In [11]:
dcfg = cfg["duplicates"]
co = pp.colocation_pairs(pts, dcfg["colocation_minute"], dcfg["colocation_cell_deg"])
co = co.merge(dup_pairs, on=["user_a", "user_b"], how="left").fillna({"shared_files": 0})
print(f"{len(co)} cặp có ít nhất một (phút, ô) chung. Phân bố share:")
print(pd.cut(co.share, [0, .001, .01, .05, .1, .2, .5, .9, 1]).value_counts().sort_index().to_string())
print("Cặp không có file trùng nhưng share cao:")
display(co[co.shared_files == 0].head(8).round(4))

422 cặp có ít nhất một (phút, ô) chung. Phân bố share:
share
(0.0, 0.001]     256
(0.001, 0.01]     61
(0.01, 0.05]      44
(0.05, 0.1]       17
(0.1, 0.2]         8
(0.2, 0.5]        15
(0.5, 0.9]         7
(0.9, 1.0]        14
Cặp không có file trùng nhưng share cao:


,user_a,user_b,shared,n_a,n_b,share,shared_files
12,49,122,1961,2034,9871,0.9641,0.0
15,48,49,1627,4843,2034,0.7999,0.0
25,48,122,1997,4843,9871,0.4123,0.0
28,163,175,97,251312,306,0.3170,0.0
29,0,30,12418,42181,92325,0.2944,0.0
30,0,4,11822,42181,112760,0.2803,0.0
31,164,176,74,279,311,0.2652,0.0
33,129,167,328,1335,79829,0.2457,0.0


In [12]:
strong = co[co.share >= dcfg["colocation_min_share"]][["user_a", "user_b"]]
edges = pd.concat([dup_pairs[["user_a", "user_b"]], strong]).drop_duplicates()
group = pp.components(pts.user.unique(), edges)
gs = group.value_counts()
print(f"{len(strong)} cặp share >= {dcfg['colocation_min_share']}; trong đó {int((co.share >= dcfg['colocation_min_share']).sum() - (co[(co.share >= dcfg['colocation_min_share']) & (co.shared_files > 0)].shape[0]))} cặp không có file trùng")
print(f"{len(gs)} nhóm; {int((gs > 1).sum())} nhóm có > 1 user, gồm {int(gs[gs > 1].sum())} user; nhóm lớn nhất {gs.max()} user")
rep["groups"] = {"dup_file_pairs": len(dup_pairs), "colocation_pairs": len(strong), "groups": len(gs),
                 "multi_user_groups": int((gs > 1).sum()), "largest_group_users": int(gs.max())}

44 cặp share >= 0.1; trong đó 15 cặp không có file trùng
139 nhóm; 19 nhóm có > 1 user, gồm 62 user; nhóm lớn nhất 20 user


## 10. Chia dev / holdout

- **Theo nhóm:** cả nhóm (bước 9) nằm cùng một phía.
- **Nhóm chiếm quá nhiều dữ liệu** (> 5% số điểm) bị ép vào `dev`: GeoLife có vài power user chiếm phần lớn dữ liệu (EDA mục 1);
  nếu họ ở holdout thì chỉ số holdout gần như chỉ phản ánh họ.
- **Phân tầng** theo (có nhãn phương tiện hay không) × (số ngày có dữ liệu, 3 khoảng), để hai phía có phân bố giống nhau và cả hai
  đều có user có nhãn (cần cho chỉ số stay giả).
- `holdout` chỉ dùng **một lần** để báo cáo, sau khi đã chốt tham số trên `dev`.

In [13]:
u = pp.user_table(pts, files, labels, group)
s = cfg["split"]
u["split"] = pp.split_groups(u, s["holdout_frac"], s["max_holdout_group_share"], s["n_strata"], cfg["seed"])
print("User chiếm nhiều điểm nhất:"); display(u.nlargest(6, "point_share")[["n_points", "point_share", "days", "dup_share", "group", "group_size", "split"]].round(3))
summ = u.groupby("split").agg(users=("n_points", "size"), groups=("group", "nunique"), labeled_users=("has_labels", "sum"),
                              points=("n_points", "sum"), median_days=("days", "median"), max_user_share=("point_share", "max"),
                              mean_dup_share=("dup_share", "mean"))
summ["point_share"] = summ.points / summ.points.sum()
display(summ.round(3))

User chiếm nhiều điểm nhất:


,n_points,point_share,days,dup_share,group,group_size,split
user,,,,,,,
153,2156929,0.090,1252,0.309,51,20,dev
128,1199044,0.050,909,0.033,51,20,dev
41,1057033,0.044,138,0.000,37,1,holdout
17,1026174,0.043,195,0.000,15,1,dev
163,967830,0.040,559,0.645,51,20,dev
68,937856,0.039,295,0.000,62,1,dev


,users,groups,labeled_users,points,median_days,max_user_share,mean_dup_share,point_share
split,,,,,,,,
dev,135,100,53,18452398,24.0,0.090,0.131,0.766
holdout,47,39,16,5639032,18.0,0.044,0.156,0.234


In [14]:
# Kiểm tra rò rỉ
cross_groups = int(u.groupby("group").split.nunique().gt(1).sum())
side = u.split
co["side_a"], co["side_b"] = side.reindex(co.user_a).to_numpy(), side.reindex(co.user_b).to_numpy()
cross = co[co.side_a != co.side_b]
print("Nhóm nằm ở cả hai phía:", cross_groups)
print("Cặp có file trùng nằm ở hai phía:", int((cross.shared_files > 0).sum()))
print(f"Cặp có (phút, ô) chung nằm ở hai phía: {len(cross)}, share lớn nhất {cross.share.max():.4f}")
display(cross.head(5).round(4))
rep["split"] = {"summary": summ.reset_index().to_dict(orient="records"), "cross_groups": cross_groups,
                "cross_pairs_max_share": float(cross.share.max()) if len(cross) else 0.0}

Nhóm nằm ở cả hai phía: 0
Cặp có file trùng nằm ở hai phía: 0
Cặp có (phút, ô) chung nằm ở hai phía: 126, share lớn nhất 0.0525


,user_a,user_b,shared,n_a,n_b,share,shared_files,side_a,side_b
58,3,5,896,126541,17061,0.0525,0.0,dev,holdout
59,4,15,810,112760,15696,0.0516,0.0,dev,holdout
62,101,174,207,4196,4527,0.0493,0.0,dev,holdout
63,4,5,825,112760,17061,0.0484,0.0,dev,holdout
69,153,155,281,393734,7132,0.0394,0.0,dev,holdout


## 11. Ghi output

| File | Nội dung | Ai dùng |
| --- | --- | --- |
| `points.parquet` | `user, traj_id, ts_utc, ts_local, tz, lat, lon`, sort theo `(user, ts_utc)` | Model (input duy nhất) |
| `labels.parquet` | Nhãn phương tiện gốc: `user, start_utc, end_utc, mode` | Chỉ đánh giá (stay giả) |
| `users.parquet` | `n_points, n_trajs, days, point_share, has_labels, dup_share, group, group_size, split` | Chọn user theo split |
| `report.json` | Config đã dùng, số liệu từng bước, file bị loại kèm lý do | Kiểm tra / tái lập |

In [15]:
pts["traj_id"] = pts.traj_id.astype("category")
pts[["user", "traj_id", "ts_utc", "ts_local", "tz", "lat", "lon"]].to_parquet(OUT / "points.parquet", index=False)
labels.to_parquet(OUT / "labels.parquet", index=False)
u.to_parquet(OUT / "users.parquet")
files["status"] = "kept"
files.loc[files.traj_key.isin(flight), "status"] = "dropped_flight"
files.loc[files.traj_key.isin(corrupted), "status"] = "dropped_corrupted"
kept = set(pts.user.astype(str) + "_" + pts.traj_id.astype(str))
files.loc[(files.status == "kept") & ~files.traj_key.isin(kept), "status"] = "dropped_empty"
rep["config"] = cfg
rep["dropped_files"] = files[files.status != "kept"][["traj_key", "status"]].to_dict(orient="records")
rep["clean"] = {"points": len(pts), "trajs": len(kept), "users": int(pts.user.nunique()),
                "kept_share": round(len(pts) / rep["raw"]["points"], 4)}
rep["runtime_s"] = round(time.time() - t0, 1)
(OUT / "report.json").write_text(json.dumps(rep, indent=1, ensure_ascii=False, default=lambda x: x.item() if hasattr(x, "item") else str(x)))
for f in ["points.parquet", "labels.parquet", "users.parquet", "report.json"]:
    print(f"{f:16s} {(OUT / f).stat().st_size / 1e6:7.1f} MB")
print(rep["clean"], "| runtime", rep["runtime_s"], "s")

points.parquet     604.8 MB
labels.parquet       0.2 MB
users.parquet        0.0 MB
report.json          0.0 MB
{'points': 24091430, 'trajs': 18649, 'users': 182, 'kept_share': 0.9684} | runtime 78.7 s
